# Step 2 check on Kaggle

Confirms that both baselines learn on a small set and use one evaluation, and measures their speed.
Small-run accuracy here is not a result.

Use Accelerator **GPU T4 x2**: each model runs on its own GPU. Before running, attach your Kaggle
Dataset and turn **Internet on**. Stop the session when it finishes, so it stops using quota.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
!PYTHONPATH=src python -m pytest tests/test_exp_setup.py tests/test_baseline.py -q


In [ ]:
!PYTHONPATH=src python -m quickdraw_exp.check_baseline


In [ ]:
import json

runs = sorted(Path("/kaggle/working/runs").glob("*_check_baseline_s*"))
summary = json.loads((runs[-1] / "summary.json").read_text())
env = json.loads((runs[-1] / "env.json").read_text())
print("PASSED" if summary["passed"] else "FAILED")
print(json.dumps({k: summary[k] for k in ("crashed", "same_evaluation", "gpus", "torch", "budget")}, indent=2))
for kind, result in summary["results"].items():
    print(kind, json.dumps({k: result[k] for k in ("memorise_accuracy", "memorise_seconds", "small_run_seconds", "small_run", "val_top1_by_frac", "checks")}, indent=2))
print(json.dumps({k: env[k] for k in ("git_commit", "git_dirty", "account")}, indent=2))
